Purpose: Examine results of expression partitioning using HybridExpress.<br>
Author: Anna Pardo<br>
Date initiated: Feb. 15, 2026

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

In [12]:
# load results dataframe
res = pd.read_csv("./hybexp_partitiongenes_bygttreat.txt",sep="\t",header="infer")
res.head()

,Gene,Category,Class,lFC_F1_vs_P1,lFC_F1_vs_P2,genotype_treat
0,recip_syn1000,1,ADD,6.110441,-0.780159,18_D
1,recip_syn10003,1,ADD,3.313729,-0.887941,18_D
2,recip_syn10008,1,ADD,1.461161,-0.976261,18_D
3,recip_syn10061,1,ADD,2.826917,-1.349656,18_D
4,recip_syn10063,1,ADD,1.728989,-2.551956,18_D


In [13]:
res["Class"].unique()

array(['ADD', 'ELD_P2', 'DOWN', 'ELD_P1', 'UP'], dtype=object)

In [17]:
# split genotype & treatment info
gtt = res["genotype_treat"].str.split("_",expand=True)
gtt.head()

,0,1
0,18,D
1,18,D
2,18,D
3,18,D
4,18,D


In [19]:
gtt.rename(columns={0:"genotype",1:"treat"},inplace=True)
res = pd.concat([res,gtt],axis=1)
res.head()

,Gene,Category,Class,lFC_F1_vs_P1,lFC_F1_vs_P2,genotype_treat,0,1,genotype,treat
0,recip_syn1000,1,ADD,6.110441,-0.780159,18_D,18,D,18,D
1,recip_syn10003,1,ADD,3.313729,-0.887941,18_D,18,D,18,D
2,recip_syn10008,1,ADD,1.461161,-0.976261,18_D,18,D,18,D
3,recip_syn10061,1,ADD,2.826917,-1.349656,18_D,18,D,18,D
4,recip_syn10063,1,ADD,1.728989,-2.551956,18_D,18,D,18,D


In [5]:
# note this is the results from both treatments combined.
# note also: P1 = aloifolia, P2 = filamentosa
# ELD = expression-level dominance

# find the genes that are in the same class (with and without being in the same category) between genotypes
## we will want to know about genotype physiology classification, so load that information

physcat = json.load(open("/home/leviathan22/Yucca_genomics/phys_figures/physiological_CAM_categories.json"))
physcat

{'18': 'C3',
 '48': 'facultative_CAM',
 '43': 'facultative_CAM',
 '70': 'C3',
 '61': 'C3',
 '55': 'C3',
 '1AB': 'possible_fac_CAM',
 '52': 'possible_fac_CAM',
 '16': 'facultative_CAM',
 '56': 'C3',
 '53': 'possible_fac_CAM',
 '51': 'possible_fac_CAM',
 '37': 'possible_fac_CAM',
 'G': 'C3',
 '45': 'C3',
 '19': 'facultative_CAM',
 '13': 'C3',
 'Eudy': 'C3',
 '46': 'possible_fac_CAM',
 '15': 'facultative_CAM',
 '2AB': 'possible_fac_CAM',
 '36': 'C3'}

In [22]:
res["phys"] = res["genotype"].map(physcat)
res.head()

,Gene,Category,Class,lFC_F1_vs_P1,lFC_F1_vs_P2,genotype_treat,genotype,treat,phys
0,recip_syn1000,1,ADD,6.110441,-0.780159,18_D,18,D,C3
1,recip_syn10003,1,ADD,3.313729,-0.887941,18_D,18,D,C3
2,recip_syn10008,1,ADD,1.461161,-0.976261,18_D,18,D,C3
3,recip_syn10061,1,ADD,2.826917,-1.349656,18_D,18,D,C3
4,recip_syn10063,1,ADD,1.728989,-2.551956,18_D,18,D,C3


In [23]:
# convert to two dictionaries: one each for class & category with the following structure
## key=class/category, value = {key=genotype, value=gene list}

def make_genelists_dict(colname):
    outdict = {}
    for i in res[colname].unique():
        df = res[res[colname]==i]
        outdict[i] = {}
        for g in df["genotype_treat"].unique():
            gdf = df[df["genotype_treat"]==g]
            outdict[i][g] = list(gdf["Gene"].unique())
    return outdict

In [24]:
genes_by_class = make_genelists_dict("Class")

In [25]:
genes_by_cat = make_genelists_dict("Category")

In [26]:
genes_by_cat.keys()

dict_keys([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12])

In [34]:
a = [[1,3,5,3],[1,6,78,5],[3,4,5]]

In [36]:
set.intersection(*map(set,a))

{5}

In [54]:
# make a function to get the intersection of all given genotypes for a given treatment
def get_int_all(gtlist,treat,indict):
    # where indict = a subdict of genes_by_cat or genes_by_class
    
    if treat != "both":
        gtl = []
        for i in gtlist:
            gtl.append(i+"_"+treat)
    else:
        gtl = []
        for i in gtlist:
            for j in ["W","D"]:
                gtl.append(i+"_"+j)
        
    # subset indict
    d = {k: indict[k] for k in gtl if k in indict.keys()}
    
    # get list of values & run set.intersection
    intlist = list(set.intersection(*map(set,list(d.values()))))
    return {"genotypes":list(d.keys()),"shared_syntelogs":intlist,"n_shared":len(intlist)}

In [40]:
res["phys"].unique()

array(['C3', 'possible_fac_CAM', 'facultative_CAM'], dtype=object)

In [42]:
# make some genotype lists of interest
allgts = list(res["genotype"].unique())
c3s = list(res[res["phys"]=="C3"]["genotype"].unique())
pfcs = list(res[res["phys"]=="possible_fac_CAM"]["genotype"].unique())
fcs = list(res[res["phys"]=="facultative_CAM"]["genotype"].unique())

In [55]:
# get intersection of all genotypes (control & drought) - by category (more restrictive)
## note 4 genotypes missing from drought: 15, Eudy, 53, 48

allcatw = {}
allcatd = {}
allcatboth = {}
for k in genes_by_cat.keys():
    allcatw[k] = get_int_all(allgts,"W",genes_by_cat[k])
    allcatd[k] = get_int_all(allgts,"D",genes_by_cat[k])
    allcatboth[k] = get_int_all(allgts,"both",genes_by_cat[k])

In [52]:
pd.DataFrame(allcatd)

,1,2,3,4,5,6,7,8,9,10,11,12
genotypes,"[18_D, 2AB_D, 1AB_D, 19_D, G_D, 56_D, 36_D, 13...","[18_D, 2AB_D, 1AB_D, 19_D, G_D, 56_D, 36_D, 13...","[18_D, 2AB_D, 1AB_D, 19_D, G_D, 56_D, 36_D, 13...","[18_D, 2AB_D, 1AB_D, 19_D, G_D, 56_D, 36_D, 13...","[18_D, 2AB_D, 1AB_D, 19_D, G_D, 56_D, 36_D, 13...","[18_D, 2AB_D, 1AB_D, 19_D, G_D, 56_D, 36_D, 13...","[18_D, 2AB_D, 1AB_D, 19_D, G_D, 56_D, 36_D, 13...","[18_D, 2AB_D, 1AB_D, 19_D, G_D, 56_D, 36_D, 13...","[18_D, 2AB_D, 1AB_D, 19_D, G_D, 56_D, 36_D, 13...","[18_D, 2AB_D, 1AB_D, 19_D, G_D, 56_D, 36_D, 13...","[18_D, 2AB_D, 1AB_D, 19_D, G_D, 56_D, 36_D, 13...","[18_D, 2AB_D, 1AB_D, 19_D, G_D, 56_D, 36_D, 13..."
shared_syntelogs,"[recip_syn24804, recip_syn21884, recip_syn1493...","[recip_syn6381, recip_syn21822, recip_syn9422,...","[recip_syn28918, recip_syn3610, recip_syn14109...","[recip_syn4180, recip_syn18988, recip_syn22870...","[recip_syn7411, recip_syn18274, recip_syn25999...","[recip_syn5382, recip_syn6790]","[recip_syn2768, recip_syn29328, recip_syn2572,...","[recip_syn12725, recip_syn32433, recip_syn1412...","[recip_syn21974, recip_syn2104, recip_syn12024...","[recip_syn15705, recip_syn21685, recip_syn7385...","[recip_syn23833, recip_syn16238, recip_syn2674...","[recip_syn31692, recip_syn21717, recip_syn2702..."
n_shared,407,56,13,49,6,2,87,52,40,43,30,145


In [53]:
pd.DataFrame(allcatw)

,1,2,3,4,5,6,7,8,9,10,11,12
genotypes,"[18_W, 2AB_W, 1AB_W, 19_W, G_W, 56_W, 36_W, 13...","[18_W, 2AB_W, 1AB_W, 19_W, G_W, 56_W, 36_W, 13...","[18_W, 2AB_W, 1AB_W, 19_W, G_W, 56_W, 36_W, 13...","[18_W, 2AB_W, 1AB_W, 19_W, G_W, 56_W, 36_W, 13...","[18_W, 2AB_W, 1AB_W, 19_W, G_W, 56_W, 36_W, 13...","[18_W, 2AB_W, 1AB_W, 19_W, G_W, 56_W, 36_W, 13...","[18_W, 2AB_W, 1AB_W, 19_W, G_W, 56_W, 36_W, 13...","[18_W, 2AB_W, 1AB_W, 19_W, G_W, 56_W, 36_W, 13...","[18_W, 2AB_W, 1AB_W, 19_W, G_W, 56_W, 36_W, 13...","[18_W, 2AB_W, 1AB_W, 19_W, G_W, 56_W, 36_W, 13...","[18_W, 2AB_W, 1AB_W, 19_W, G_W, 56_W, 36_W, 13...","[18_W, 2AB_W, 1AB_W, 19_W, G_W, 56_W, 36_W, 13..."
shared_syntelogs,"[recip_syn24804, recip_syn21884, recip_syn1935...","[recip_syn20990, recip_syn15416, recip_syn7360...","[recip_syn3437, recip_syn29864, recip_syn12437...","[recip_syn2431, recip_syn1576, recip_syn26523,...","[recip_syn12648, recip_syn5813, recip_syn13508...","[recip_syn14513, recip_syn27367, recip_syn2350...","[recip_syn2048, recip_syn29328, recip_syn20782...","[recip_syn28062, recip_syn9408, recip_syn18323...","[recip_syn28320, recip_syn30901, recip_syn2558...","[recip_syn15705, recip_syn19973, recip_syn5973...","[recip_syn12701, recip_syn15866, recip_syn2551...","[recip_syn3184, recip_syn15830, recip_syn31692..."
n_shared,402,17,20,23,4,8,99,22,36,32,20,168


In [56]:
pd.DataFrame(allcatboth)

,1,2,3,4,5,6,7,8,9,10,11,12
genotypes,"[18_W, 18_D, 2AB_W, 2AB_D, 1AB_W, 1AB_D, 19_W,...","[18_W, 18_D, 2AB_W, 2AB_D, 1AB_W, 1AB_D, 19_W,...","[18_W, 18_D, 2AB_W, 2AB_D, 1AB_W, 1AB_D, 19_W,...","[18_W, 18_D, 2AB_W, 2AB_D, 1AB_W, 1AB_D, 19_W,...","[18_W, 18_D, 2AB_W, 2AB_D, 1AB_W, 1AB_D, 19_W,...","[18_W, 18_D, 2AB_W, 2AB_D, 1AB_W, 1AB_D, 19_W,...","[18_W, 18_D, 2AB_W, 2AB_D, 1AB_W, 1AB_D, 19_W,...","[18_W, 18_D, 2AB_W, 2AB_D, 1AB_W, 1AB_D, 19_W,...","[18_W, 18_D, 2AB_W, 2AB_D, 1AB_W, 1AB_D, 19_W,...","[18_W, 18_D, 2AB_W, 2AB_D, 1AB_W, 1AB_D, 19_W,...","[18_W, 18_D, 2AB_W, 2AB_D, 1AB_W, 1AB_D, 19_W,...","[18_W, 18_D, 2AB_W, 2AB_D, 1AB_W, 1AB_D, 19_W,..."
shared_syntelogs,"[recip_syn24804, recip_syn24926, recip_syn2188...","[recip_syn15416, recip_syn7360, recip_syn27790...","[recip_syn28918, recip_syn17391, recip_syn1243...",[recip_syn30023],[recip_syn7411],[recip_syn6790],"[recip_syn30047, recip_syn6627, recip_syn4948,...","[recip_syn4570, recip_syn31005, recip_syn32140...","[recip_syn13820, recip_syn11738, recip_syn1716...","[recip_syn15705, recip_syn2564, recip_syn15367...","[recip_syn25513, recip_syn14424, recip_syn29111]","[recip_syn31692, recip_syn27029, recip_syn3001..."
n_shared,255,5,4,1,1,1,13,8,12,18,3,83


In [57]:
# repeat for each physiological genotype set

c3cat = {"D":{},"W":{},"both":{}}
for k in genes_by_cat.keys():
    for i in c3cat.keys():
        c3cat[i][k] = get_int_all(c3s,i,genes_by_cat[k])

In [62]:
pd.DataFrame(c3cat["both"])

,1,2,3,4,5,6,7,8,9,10,11,12
genotypes,"[18_W, 18_D, G_W, G_D, 56_W, 56_D, 36_W, 36_D,...","[18_W, 18_D, G_W, G_D, 56_W, 56_D, 36_W, 36_D,...","[18_W, 18_D, G_W, G_D, 56_W, 56_D, 36_W, 36_D,...","[18_W, 18_D, G_W, G_D, 56_W, 56_D, 36_W, 36_D,...","[18_W, 18_D, G_W, G_D, 56_W, 56_D, 36_W, 36_D,...","[18_W, 18_D, G_W, G_D, 56_W, 56_D, 36_W, 36_D,...","[18_W, 18_D, G_W, G_D, 56_W, 56_D, 36_W, 36_D,...","[18_W, 18_D, G_W, G_D, 56_W, 56_D, 36_W, 36_D,...","[18_W, 18_D, G_W, G_D, 56_W, 56_D, 36_W, 36_D,...","[18_W, 18_D, G_W, G_D, 56_W, 56_D, 36_W, 36_D,...","[18_W, 18_D, G_W, G_D, 56_W, 56_D, 36_W, 36_D,...","[18_W, 18_D, G_W, G_D, 56_W, 56_D, 36_W, 36_D,..."
shared_syntelogs,"[recip_syn24804, recip_syn21884, recip_syn1493...","[recip_syn23773, recip_syn31978, recip_syn2420...","[recip_syn28918, recip_syn1638, recip_syn12437...","[recip_syn5259, recip_syn8040, recip_syn3620, ...","[recip_syn25721, recip_syn7411]",[recip_syn6790],"[recip_syn30047, recip_syn30502, recip_syn2932...","[recip_syn26542, recip_syn4570, recip_syn18172...","[recip_syn20191, recip_syn25581, recip_syn1382...","[recip_syn15705, recip_syn2564, recip_syn15367...","[recip_syn19337, recip_syn14424, recip_syn3160...","[recip_syn30408, recip_syn31692, recip_syn2702..."
n_shared,376,24,8,15,2,1,33,15,29,22,9,122


In [63]:
pfccat = {"D":{},"W":{},"both":{}}
for k in genes_by_cat.keys():
    for i in pfccat.keys():
        pfccat[i][k] = get_int_all(pfcs,i,genes_by_cat[k])

In [64]:
fccat = {"D":{},"W":{},"both":{}}
for k in genes_by_cat.keys():
    for i in fccat.keys():
        fccat[i][k] = get_int_all(fcs,i,genes_by_cat[k])

In [65]:
# repeat by class

allclass = {"D":{},"W":{},"both":{}}
for k in genes_by_class.keys():
    for i in allclass.keys():
        allclass[i][k] = get_int_all(allgts,i,genes_by_class[k])

In [68]:
pd.DataFrame(allclass["both"])

,ADD,ELD_P2,DOWN,ELD_P1,UP
genotypes,"[18_W, 18_D, 2AB_W, 2AB_D, 1AB_W, 1AB_D, 19_W,...","[18_W, 18_D, 2AB_W, 2AB_D, 1AB_W, 1AB_D, 19_W,...","[18_W, 18_D, 2AB_W, 2AB_D, 1AB_W, 1AB_D, 19_W,...","[18_W, 18_D, 2AB_W, 2AB_D, 1AB_W, 1AB_D, 19_W,...","[18_W, 18_D, 2AB_W, 2AB_D, 1AB_W, 1AB_D, 19_W,..."
shared_syntelogs,"[recip_syn24804, recip_syn21884, recip_syn1493...","[recip_syn15416, recip_syn7360, recip_syn14424...","[recip_syn30047, recip_syn15705, recip_syn1997...","[recip_syn13820, recip_syn11738, recip_syn1716...","[recip_syn6437, recip_syn4570, recip_syn31005,..."
n_shared,338,8,61,13,12


In [69]:
c3class = {"D":{},"W":{},"both":{}}
for k in genes_by_class.keys():
    for i in c3class.keys():
        c3class[i][k] = get_int_all(c3s,i,genes_by_class[k])

In [70]:
pfcclass = {"D":{},"W":{},"both":{}}
for k in genes_by_class.keys():
    for i in pfcclass.keys():
        pfcclass[i][k] = get_int_all(pfcs,i,genes_by_class[k])

In [71]:
fcclass = {"D":{},"W":{},"both":{}}
for k in genes_by_class.keys():
    for i in fcclass.keys():
        fcclass[i][k] = get_int_all(fcs,i,genes_by_class[k])

In [72]:
pd.DataFrame(fcclass["W"])

,ADD,ELD_P2,DOWN,ELD_P1,UP
genotypes,"[19_W, 43_W, 15_W, 48_W]","[19_W, 43_W, 15_W, 48_W]","[19_W, 43_W, 15_W, 48_W]","[19_W, 43_W, 15_W, 48_W]","[19_W, 43_W, 15_W, 48_W]"
shared_syntelogs,"[recip_syn15830, recip_syn20266, recip_syn5729...","[recip_syn2194, recip_syn11950, recip_syn27216...","[recip_syn10722, recip_syn2768, recip_syn10756...","[recip_syn4658, recip_syn21041, recip_syn7549,...","[recip_syn12345, recip_syn22347, recip_syn8650..."
n_shared,1281,890,693,1227,364
